In [ ]:
import numpy as np

In [ ]:
class LogisticRegression:
    def __init__(self, eta, n_iterations, threshold=0.5):
        self.eta = eta
        self.n_iterations = n_iterations
        self.threshold = threshold
        self.w = None
        self.losses = None
        self.n_features = None

    def fit(self, X, y):
        X = np.asarray(X)
        if X.ndim != 2:
            raise ValueError
        y = np.asarray(y).ravel()
        if X.shape[0] != y.shape[0]:
            raise ValueError
        d = X.shape[1]
        self.n_features = d
        X_full = np.column_stack([np.ones(len(X)), X])
        self.w = np.zeros(d + 1)
        self.losses = []
        l = len(X)
        for i in range(self.n_iterations):
            z = X_full @ self.w
            p = 1 / (1 + np.exp(-z))
            L = -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
            g = (1 / l) * (X_full.T @ (p - y))
            self.w = self.w - self.eta * g
            self.losses.append(L)
        return self

    def predict_proba(self, X):
        if self.w is None:
            raise ValueError

        X = np.asarray(X)
        if X.ndim != 2:
            raise ValueError
        if X.shape[1] != self.n_features:
            raise ValueError

        X_full = np.column_stack([np.ones(len(X)), X])
        z = X_full @ self.w
        return 1 / (1 + np.exp(-z))

    def predict(self, X, threshold=None):
        th = self.threshold if threshold is None else threshold
        p = self.predict_proba(X)
        return (p >= th).astype(int)

    def score(self, X, y, threshold=None):
        y_pred = self.predict(X, threshold=threshold)

        y = np.asarray(y)
        if y.ndim == 0 or y.ndim > 2:
            raise ValueError
        if y.ndim == 2:
            if y.shape[0] > 1 and y.shape[1] > 1:
                raise ValueError
            y = y.reshape(-1)

        if y.shape[0] != y_pred.shape[0]:
            raise ValueError

        tp = np.sum((y == 1) & (y_pred == 1))
        tn = np.sum((y == 0) & (y_pred == 0))
        fp = np.sum((y == 0) & (y_pred == 1))
        fn = np.sum((y == 1) & (y_pred == 0))

        accuracy = (tp + tn) / (tp + tn + fp + fn)

        if tp + fp == 0:
            precision = 0.0
        else:
            precision = tp / (tp + fp)

        if tp + fn == 0:
            recall = 0.0
        else:
            recall = tp / (tp + fn)

        if precision + recall == 0:
            f1 = 0.0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        confusion_matrix = np.array([[tp, fp], [fn, tn]])

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'confusion_matrix': confusion_matrix,
        }

In [ ]:
from sklearn import datasets

def _dummy_model():
    """Заготовка: предсказание = 1 если x=1, иначе 0."""
    m = LogisticRegression(eta=0.05, n_iterations=1, threshold=0.5)
    m.n_features = 1
    m.w = np.array([-5.0, 10.0])  # bias, coef
    return m

# ============================================================
# T1 — пример a1 из ноутбука 18 (с исправлением опечатки)
# Матрица: [[45, 5], [5, 45]]  (TP, FP; FN, TN)
# Ожидается: A = P = R = F1 = 0.9
# (В ноутбуке опечатка: P=R=0.8; правильные значения — 0.9)
# ============================================================
y_true = np.array([1]*50 + [0]*50)
X      = np.array([[1]]*45 + [[0]]*5 + [[1]]*5 + [[0]]*45, dtype=float)
r = _dummy_model().score(X, y_true)
print("T1 (a1):")
print(f"  accuracy  = {r['accuracy']:.4f}   ожидание: 0.9")
print(f"  precision = {r['precision']:.4f}   ожидание: 0.9")
print(f"  recall    = {r['recall']:.4f}   ожидание: 0.9")
print(f"  f1        = {r['f1']:.4f}   ожидание: 0.9")
print(f"  CM = {r['confusion_matrix'].tolist()}   ожидание: [[45,5],[5,45]]")

# ============================================================
# T2 — пример a2 из ноутбука 18 (значения в ноутбуке верны)
# Матрица: [[12, 8], [20, 60]]
# Ожидается: A=0.72, P=0.6, R=0.375, F1≈0.4615
# ============================================================
y_true = np.array([1]*32 + [0]*68)
X      = np.array([[1]]*12 + [[0]]*20 + [[1]]*8 + [[0]]*60, dtype=float)
r = _dummy_model().score(X, y_true)
print("\nT2 (a2):")
print(f"  accuracy  = {r['accuracy']:.4f}   ожидание: 0.72")
print(f"  precision = {r['precision']:.4f}   ожидание: 0.6")
print(f"  recall    = {r['recall']:.4f}   ожидание: 0.375")
print(f"  f1        = {r['f1']:.4f}   ожидание: ~0.4615")
print(f"  CM = {r['confusion_matrix'].tolist()}   ожидание: [[12,8],[20,60]]")

# ============================================================
# T3 — идеальный классификатор
# Ожидается: всё = 1.0
# ============================================================
y_true = np.array([0, 0, 1, 1, 1, 0])
X      = np.array([[0], [0], [1], [1], [1], [0]], dtype=float)
r = _dummy_model().score(X, y_true)
print("\nT3 (идеальный):")
print(f"  accuracy={r['accuracy']:.4f}, precision={r['precision']:.4f}, "
      f"recall={r['recall']:.4f}, f1={r['f1']:.4f}   ожидание: все 1.0")

# ============================================================
# T4 — «всегда 0» (проверка краевых защит в precision/recall/f1)
# Ожидается: A=0.5, P=R=F1=0.0 (без nan)
# ============================================================
y_true = np.array([0, 0, 1, 1, 0, 1])
X      = np.zeros((6, 1), dtype=float)  # все предсказания = 0
r = _dummy_model().score(X, y_true)
print("\nT4 (всегда 0):")
print(f"  accuracy  = {r['accuracy']:.4f}   ожидание: 0.5")
print(f"  precision = {r['precision']:.4f}   ожидание: 0.0 (краевой)")
print(f"  recall    = {r['recall']:.4f}   ожидание: 0.0")
print(f"  f1        = {r['f1']:.4f}   ожидание: 0.0")
print(f"  CM = {r['confusion_matrix'].tolist()}   ожидание: [[0,0],[3,3]]")

# ============================================================
# T5 — демо-датасет из ноутбука 17
# Проверяем, что модель учится: losses убывают, метрики высокие
# ВАЖНО: точные числа ноутбука (98.571 / 96.667 и w = [3.72, 0.22])
# НЕ совпадут — у нас есть bias, у ноутбука его нет.
# ============================================================
classes = datasets.make_classification(
    n_samples=100, n_features=2, n_informative=2,
    n_redundant=0, n_classes=2, random_state=1,
)
np.random.seed(12)
idx = np.random.permutation(classes[0].shape[0])
X_sh, y_sh = classes[0][idx], classes[1][idx]
cut = int(len(classes[0]) * 0.7)
X_train, X_test = X_sh[:cut], X_sh[cut:]
y_train, y_test = y_sh[:cut], y_sh[cut:]

model = LogisticRegression(eta=0.05, n_iterations=1000, threshold=0.5)
model.fit(X_train, y_train)

train = model.score(X_train, y_train)
test  = model.score(X_test,  y_test)

print("\nT5 (демо из ноутбука 17):")
print(f"  loss[0]   = {model.losses[0]:.4f}")
print(f"  loss[-1]  = {model.losses[-1]:.4f}   (должно быть меньше loss[0])")
print(f"  w         = {model.w}   (3 компонента: bias + 2 признака)")
print(f"  train acc = {train['accuracy']:.4f}   (порядка 0.95+)")
print(f"  test  acc = {test['accuracy']:.4f}   (порядка 0.95+)")
print(f"  test  CM  = {test['confusion_matrix'].tolist()}")